In [ ]:
import os
import sys
from pathlib import Path

IN_KAGGLE = os.path.exists("/kaggle") or "KAGGLE_URL_BASE" in os.environ

if IN_KAGGLE:
    # Kaggle usually has torch/transformers preinstalled; install peft if missing.
    try:
        import peft  # noqa: F401
    except Exception:
        %pip -q install peft

PROJECT_ROOT = os.environ.get("PROJECT_ROOT", "")
if PROJECT_ROOT:
    print(f"Using PROJECT_ROOT from environment: {PROJECT_ROOT}")

In [ ]:
import json
import sys
from pathlib import Path

# Resolve project root in local and Kaggle environments.
candidate_roots = []

# 1) Explicit env override from setup cell
project_root_env = Path(PROJECT_ROOT).expanduser() if "PROJECT_ROOT" in globals() and PROJECT_ROOT else None
if project_root_env:
    candidate_roots.append(project_root_env)

# 2) Current working directory search upward
cwd = Path.cwd().resolve()
candidate_roots.extend([cwd, *cwd.parents])

# 3) Notebook location (when available)
try:
    nb_dir = Path(__file__).resolve().parent
    candidate_roots.extend([nb_dir, *nb_dir.parents])
except NameError:
    pass

# 4) Common Kaggle locations
candidate_roots.extend([
    Path("/kaggle/working/Legal-question-answer-v1"),
    Path("/kaggle/working"),
    Path("/kaggle/input/Legal-question-answer-v1"),
])

def is_project_root(path: Path) -> bool:
    return (path / "utils").exists() and (path / "data").exists()

project_root = None
seen = set()
for path in candidate_roots:
    resolved = path.resolve()
    if str(resolved) in seen:
        continue
    seen.add(str(resolved))
    if is_project_root(resolved):
        project_root = resolved
        break

print(f"Resolved project_root: {project_root}")

def get_jsonl_data(file_path: str):
    rows = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON at line {line_no} in {file_path}: {e}") from e
    return rows

In [ ]:
# Explicit Kaggle dataset paths
KAGGLE_CORPUS_PATH = Path("/kaggle/input/datasets/trinhthaison/corpus-legal-answer-question-v1/corpus.jsonl")
KAGGLE_CHUNK_STORE_PATH = Path("/kaggle/input/datasets/trinhthaison/chunk-store-legal-answer-question-v1/chunk_store.jsonl")

# Optional explicit overrides for non-Kaggle runs
corpus_override = os.environ.get("CORPUS_JSONL_PATH", "").strip()
chunk_store_override = os.environ.get("CHUNK_STORE_JSONL_PATH", "").strip()

if corpus_override and chunk_store_override:
    corpus_path = Path(corpus_override).expanduser().resolve()
    chunk_store_path = Path(chunk_store_override).expanduser().resolve()
elif "IN_KAGGLE" in globals() and IN_KAGGLE:
    corpus_path = KAGGLE_CORPUS_PATH
    chunk_store_path = KAGGLE_CHUNK_STORE_PATH
else:
    # Local default
    corpus_path = project_root / "data" / "corpus.jsonl"
    chunk_store_path = project_root / "data" / "chunk_store.jsonl"

if not corpus_path.exists():
    raise FileNotFoundError(f"corpus.jsonl not found: {corpus_path}")
if not chunk_store_path.exists():
    raise FileNotFoundError(f"chunk_store.jsonl not found: {chunk_store_path}")

print(f"Using corpus_path: {corpus_path}")
print(f"Using chunk_store_path: {chunk_store_path}")

corpus = get_jsonl_data(str(corpus_path))
chunk_store = get_jsonl_data(str(chunk_store_path))

print(corpus[0])
print(chunk_store[0])

In [ ]:
from collections import defaultdict
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

try:
    from peft import LoraConfig, TaskType, get_peft_model
    PEFT_AVAILABLE = True
except Exception:
    PEFT_AVAILABLE = False

def normalize_whitespace(text):
    return " ".join((text or "").split())

class DualEncoder(torch.nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.q_encoder = AutoModel.from_pretrained(model_name)
        self.d_encoder = AutoModel.from_pretrained(model_name)

    @staticmethod
    def mean_pool(last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        summed = torch.sum(last_hidden_state * mask, dim=1)
        denom = torch.clamp(mask.sum(dim=1), min=1e-9)
        return summed / denom

    def encode_questions(self, q_tok):
        out = self.q_encoder(
            input_ids=q_tok["input_ids"],
            attention_mask=q_tok["attention_mask"],
        )
        emb = self.mean_pool(out.last_hidden_state, q_tok["attention_mask"])
        return F.normalize(emb, p=2, dim=-1)

    def encode_docs(self, d_tok):
        out = self.d_encoder(
            input_ids=d_tok["input_ids"],
            attention_mask=d_tok["attention_mask"],
        )
        emb = self.mean_pool(out.last_hidden_state, d_tok["attention_mask"])
        return F.normalize(emb, p=2, dim=-1)


def apply_lora_if_needed(model: DualEncoder, cfg: dict):
    lora_r = int(cfg.get("lora_r", 0) or 0)
    if lora_r <= 0:
        return model

    if not PEFT_AVAILABLE:
        raise ImportError(
            "Checkpoint expects LoRA adapters but `peft` is not installed. ",
            "Install with: pip install peft (or pip install -r requirements.txt)."
        )

    lora_cfg = LoraConfig(
        task_type=TaskType.FEATURE_EXTRACTION,
        r=lora_r,
        lora_alpha=int(cfg.get("lora_alpha", 32)),
        lora_dropout=float(cfg.get("lora_dropout", 0.1)),
        target_modules=cfg.get("lora_target_modules", ["query", "key", "value"]),
        bias="none",
    )
    model.q_encoder = get_peft_model(model.q_encoder, lora_cfg)
    model.d_encoder = get_peft_model(model.d_encoder, lora_cfg)
    return model


def resolve_checkpoint_path(root: Path) -> Path:
    kaggle_checkpoint = Path("/kaggle/input/datasets/trinhthaison/dual-encoder-resume-v1-legal-question-answer-v1/best_dual_encoder_resume_v1.pt")
    local_checkpoint = root / "models" / "best_dual_encoder_resume_v1.pt"

    if "IN_KAGGLE" in globals() and IN_KAGGLE:
        if kaggle_checkpoint.exists():
            return kaggle_checkpoint
        raise FileNotFoundError(
            f"Kaggle checkpoint not found: {kaggle_checkpoint}"
        )

    if local_checkpoint.exists():
        return local_checkpoint

    raise FileNotFoundError(
        "No retriever checkpoint found. "
        f"Checked: {local_checkpoint}"
    )


def load_checkpoint_compat(path: Path):
    # Works across torch versions where `weights_only` may or may not exist.
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def batched_encode_docs(model, tokenizer, texts, max_len, device, batch_size=64):
    all_emb = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            d_tok = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_len,
                return_tensors="pt",
            )
            d_tok = {k: v.to(device) for k, v in d_tok.items()}
            d_emb = model.encode_docs(d_tok)
            all_emb.append(d_emb.cpu())
    return torch.cat(all_emb, dim=0) if all_emb else torch.empty((0, 1))


top_k = 10
top_k_pos = 3
progress_every_rows = 100

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_path = resolve_checkpoint_path(project_root)
checkpoint = load_checkpoint_compat(checkpoint_path)

cfg = checkpoint.get("config", {})
model_name = checkpoint.get("model_name") or cfg.get("model_name") or "vinai/phobert-base"
max_q_len = int(cfg.get("max_q_len", 96))
max_c_len = int(cfg.get("max_c_len", 128))

print(f"Loading checkpoint: {checkpoint_path}")
print(f"Model: {model_name} | q_len={max_q_len} | c_len={max_c_len}")

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = DualEncoder(model_name)
model = apply_lora_if_needed(model, cfg)

state_dict = checkpoint.get("model_state_dict", checkpoint)
missing, unexpected = model.load_state_dict(state_dict, strict=False)
if missing:
    print(f"Missing keys while loading: {len(missing)}")
if unexpected:
    print(f"Unexpected keys while loading: {len(unexpected)}")

model = model.to(device)
model.eval()

# Pre-index chunks by doc_id.
chunks_by_doc = defaultdict(list)
for chunk in chunk_store:
    doc_id = chunk.get("doc_id")
    article_id = chunk.get("article_id")
    chunk_text = normalize_whitespace(chunk.get("text"))
    if doc_id is not None and chunk_text:
        chunks_by_doc[doc_id].append({
            "article_id": article_id,
            "text": chunk_text,
        })

training_data = []
total_rows = len(corpus)
rows_with_chunks = 0
qa_seen = 0
qa_valid = 0

print(f"Starting encoder-based chunk retrieval for {total_rows} corpus rows...")

for row_idx, row in enumerate(corpus, start=1):
    doc_id = row.get("doc_id")
    article_id = row.get("article_id")
    chunk_items = chunks_by_doc.get(doc_id, [])

    if not chunk_items:
        if row_idx % progress_every_rows == 0 or row_idx == total_rows:
            print(
                f"[Progress] rows={row_idx}/{total_rows} | rows_with_chunks={rows_with_chunks} | "
                f"qa_seen={qa_seen} | qa_valid={qa_valid} | samples={len(training_data)}"
            )
        continue

    rows_with_chunks += 1
    qa_pairs = row.get("generated_qa_pairs", [])
    # Get question which has medium difficulty (index 1)
    easy_qa_pair = qa_pairs[1] if qa_pairs else None

    if easy_qa_pair:
        qa_seen += 1
        question = normalize_whitespace(easy_qa_pair.get("question"))
        answer = normalize_whitespace(easy_qa_pair.get("answer"))

        if not question or not answer:
            continue

        qa_valid += 1
        chunk_texts = [c["text"] for c in chunk_items]

        with torch.no_grad():
            q_tok = tokenizer(
                [question],
                padding=True,
                truncation=True,
                max_length=max_q_len,
                return_tensors="pt",
            )
            q_tok = {k: v.to(device) for k, v in q_tok.items()}
            q_emb = model.encode_questions(q_tok).cpu().squeeze(0)

        d_emb = batched_encode_docs(model, tokenizer, chunk_texts, max_c_len, device, batch_size=64)
        if d_emb.numel() == 0:
            continue

        scores = torch.matmul(d_emb, q_emb)
        k = min(top_k, scores.shape[0])
        if k <= 0:
            continue

        top_idx = torch.topk(scores, k=k, largest=True).indices.tolist()
        top_chunks = [chunk_items[i] for i in top_idx]

        pos_candidates = [c["text"] for c in top_chunks if c.get("article_id") == article_id]
        neg_candidates = [c["text"] for c in top_chunks if c.get("article_id") != article_id]

        pair_count = min(len(pos_candidates), top_k_pos)
        top_k_positive_chunks = pos_candidates[:pair_count]
        top_k_negative_chunks = neg_candidates[:pair_count]

        # If negatives are insufficient, fill from other chunks in same doc_id (excluding already used).
        if len(top_k_negative_chunks) < pair_count:
            used_texts = set(top_k_positive_chunks) | set(top_k_negative_chunks)
            ranked_all_idx = torch.argsort(scores, descending=True).tolist()
            for idx_all in ranked_all_idx:
                cand_text = chunk_items[idx_all]["text"]
                if cand_text in used_texts:
                    continue
                top_k_negative_chunks.append(cand_text)
                used_texts.add(cand_text)
                if len(top_k_negative_chunks) >= pair_count:
                    break

        sample = {
            "doc_id": doc_id,
            "article_id": article_id,
            "question": question,
            "answer": answer,
            "positive_chunks": top_k_positive_chunks,
            "negative_chunks": top_k_negative_chunks,
        }
        training_data.append(sample)

    if row_idx % progress_every_rows == 0 or row_idx == total_rows:
        print(
            f"[Progress] rows={row_idx}/{total_rows} | rows_with_chunks={rows_with_chunks} | "
            f"qa_seen={qa_seen} | qa_valid={qa_valid} | samples={len(training_data)}"
        )

print(
    f"Done. rows={total_rows}, rows_with_chunks={rows_with_chunks}, "
    f"qa_seen={qa_seen}, qa_valid={qa_valid}, samples={len(training_data)}"
)

In [ ]:
print("Number of training samples:", len(training_data))
if training_data:
    print(training_data[0])
else:
    print("No training samples were created. Check corpus/chunk quality and filtering conditions.")

In [ ]:
# Save the training data to a JSONL file
import json

output_path = project_root / "data" / "training_data_v2.jsonl"
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    for sample in training_data:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

print(f"Saved to: {output_path}")